# Complete Landsat record over Flask Glacier

Crops every Landsat Collection 2 Level 1 scene (1973 onwards) to the `flask_region.shp` area at native
resolution, and writes a catalogue accounting for each scene and band.

**Run this on CryoCloud.** The USGS HTTPS links redirect to a login page, so the data has to be read from the
requester-pays `usgs-landsat` S3 bucket, which needs AWS credentials.

Based on section 3 of the [CryoCloud IS2/Landsat tutorial](https://book.stratusgeo.org/is2-cloud-landsat-integration/#id-3-search-and-open-landsat-raster-imagery-from-the-cloud),
but with windowed `rasterio` reads instead of `xr.concat`: the sensors have different bands and resolutions, and
concat would pad every scene to the union of all the grids.

**How to use it**
1. With `TEST = True`, run everything. This processes 13 scenes covering all the sensors and writes to a separate
   `flask_region_test` prefix. Section 7 runs the checks and projects the time and size of the full run.
2. Review those results. Then set `TEST = False`, restart, and run sections 1-5 only (sections 6-7 use test-only
   checks).

A run can be interrupted and restarted: finished bands are skipped, and failed ones are tried again.

## 1. Setup

In [ ]:
import json
import math
import os
import tempfile
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import urlparse

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pystac_client
import rasterio
import s3fs
from pyproj import Transformer
from pystac_client.stac_api_io import StacApiIO
from rasterio.io import MemoryFile
from rasterio.plot import plotting_extent
from rasterio.session import AWSSession
from rasterio.windows import Window, from_bounds
from urllib3 import Retry

In [ ]:
TEST = True
N_WORKERS = 16
MAX_PASSES = 3  # scenes with failed bands are retried up to this many times per run

STAC_URL = "https://landsatlook.usgs.gov/stac-server"
COLLECTION = "landsat-c2l1"
S3_REGION = "us-west-2"  # region of the usgs-landsat bucket
OUT_ROOT = "s3://nasa-cryo-scratch/jkingslake/landsat/flask_region" + ("_test" if TEST else "")

# remote sensing/worldview/strips/region_shp/flask_region.shp, EPSG:3031 (the shapefile is not in the repo)
AOI_VERTICES_3031 = [
    (-2372982.659403335, 1239458.7218937143),
    (-2359830.496259253, 1233036.2870873753),
    (-2368281.068372857, 1215305.450325377),
    (-2383338.451411642, 1223049.2473167523),
    (-2372982.659403335, 1239458.7218937143),
]
AOI_X, AOI_Y = zip(*AOI_VERTICES_3031)
AOI_BOUNDS = (min(AOI_X), min(AOI_Y), max(AOI_X), max(AOI_Y))

# Selected by name and file type, because the STAC roles are unreliable here: VZA and SZA are labelled "data",
# while qa_pixel and qa_radsat are not.
ANGLE_BANDS = {"VAA", "VZA", "SAA", "SZA"}
QA_BANDS = {"qa_pixel", "qa_radsat"}
MTL_ASSET = "MTL.json"

# GDAL reads this directly. AWSSession below sets the same thing per thread, as in the tutorial.
os.environ["AWS_REQUEST_PAYER"] = "requester"

fs = s3fs.S3FileSystem()

## 2. Search the catalogue

No cloud, date or sun-angle filter: the aim is the complete record. `limit=100` because the API returned a 502 at
`limit=500`.

In [ ]:
to_lonlat = Transformer.from_crs("EPSG:3031", "EPSG:4326", always_xy=True)
aoi_geojson = {
    "type": "Polygon",
    "coordinates": [[list(to_lonlat.transform(x, y)) for x, y in AOI_VERTICES_3031]],
}

retry = Retry(total=6, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=None)
catalog = pystac_client.Client.open(STAC_URL, stac_io=StacApiIO(max_retries=retry))
search = catalog.search(collections=[COLLECTION], intersects=aoi_geojson, limit=100)
item_collection = search.item_collection()
all_items = sorted(item_collection, key=lambda item: item.datetime)

# Stop here rather than silently working from a partial listing
matched = search.matched()
assert matched is None or matched == len(all_items), f"API matched {matched}, received {len(all_items)}"
assert len({item.id for item in all_items}) == len(all_items), "duplicate scene ids"

print(f"{len(all_items)} scenes, {all_items[0].datetime:%Y-%m-%d} to {all_items[-1].datetime:%Y-%m-%d}")
print(Counter(item.id[:4] for item in all_items))

In [ ]:
def select_test_items(items):
    """One scene per sensor/platform ID prefix, plus the awkward cases: ETM+ before and after the SLC failure, and a night scene."""
    selected = {}
    for item in items:  # items are sorted by date, so this takes the earliest of each
        prefix = item.id[:4]
        if prefix != "LE07":
            selected.setdefault(prefix, item)
    selected["LE07 SLC-on"] = next(i for i in items if i.id[:4] == "LE07" and i.datetime.year < 2003)
    selected["LE07 SLC-off"] = next(i for i in items if i.id[:4] == "LE07" and i.datetime.year > 2003)
    selected["night"] = next(
        i for i in items
        if i.properties["landsat:wrs_path"] in ("122", "123", "124") and i.properties["view:sun_elevation"] <= 0
    )
    return sorted(selected.values(), key=lambda item: item.datetime)


run_items = select_test_items(all_items) if TEST else all_items
assert not TEST or len({item.id for item in run_items}) == 13

print(f"{len(run_items)} scenes to process, writing to {OUT_ROOT}")
if TEST:
    for item in run_items:
        cloud = item.properties.get("eo:cloud_cover", float("nan"))  # missing on some scenes
        print(f"  {item.id}  sun {item.properties['view:sun_elevation']:6.1f}  cloud {cloud:6.1f}")

## 3. Cropping functions

In [ ]:
def s3_href(asset):
    return asset.extra_fields["alternate"]["s3"]["href"]


def bands_for(item):
    """Band asset names for a scene: data bands first, then QA, because the QA bands depend on the data-band results."""
    names = [
        name for name, asset in item.assets.items()
        if asset.media_type and "geotiff" in asset.media_type and name not in ANGLE_BANDS
    ]
    return sorted(names, key=lambda name: name in QA_BANDS)


def rio_env():
    # A new boto3 session for each use, because sessions are not thread-safe and hub credentials expire
    return rasterio.Env(
        AWSSession(boto3.Session(region_name=S3_REGION), requester_pays=True),
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".TIF",
        GDAL_HTTP_MAX_RETRY="5",
        GDAL_HTTP_RETRY_DELAY="2",
    )


def aoi_window(src):
    """Whole-pixel window around AOI_BOUNDS, clipped to the raster. None if they don't overlap."""
    window = from_bounds(*AOI_BOUNDS, transform=src.transform)
    col_start = max(0, math.floor(window.col_off))
    row_start = max(0, math.floor(window.row_off))
    col_stop = min(src.width, math.ceil(window.col_off + window.width))
    row_stop = min(src.height, math.ceil(window.row_off + window.height))
    if col_stop <= col_start or row_stop <= row_start:
        return None
    return Window(col_start, row_start, col_stop - col_start, row_stop - row_start)


def blocks_touched(src, window):
    """Number of internal COG blocks a window overlaps, used to estimate S3 requests."""
    block_h, block_w = src.block_shapes[0]
    n_cols = (window.col_off + window.width - 1) // block_w - window.col_off // block_w + 1
    n_rows = (window.row_off + window.height - 1) // block_h - window.row_off // block_h + 1
    return int(n_cols * n_rows)


def crop_band(href, out_path, keep_empty):
    """
    Crop one band to out_path with no resampling. Returns a record dict.

    If the crop has no valid pixels, no file is written and the status is no_data, unless keep_empty is set. QA bands
    use keep_empty, because a qa_radsat value of 0 means "not saturated" rather than "no data".
    """
    with rasterio.open(href) as src:
        window = aoi_window(src)
        if window is None:
            return {"status": "no_data", "reason": "outside_raster", "res": src.res[0]}

        data = src.read(1, window=window)
        nodata = src.nodata if src.nodata is not None else 0
        transform = src.window_transform(window)
        record = {
            "status": "ok",
            "reason": None,
            "valid_frac": float((data != nodata).mean()),
            "width": window.width,
            "height": window.height,
            "res": src.res[0],
            "x_origin": transform.c,
            "y_origin": transform.f,
            "crs": str(src.crs),
            "dtype": src.dtypes[0],
            "nodata": src.nodata,
            "blocks": blocks_touched(src, window),
        }
        if record["valid_frac"] == 0 and not keep_empty:
            return {**record, "status": "no_data", "reason": "all_nodata"}

        profile = src.profile.copy()
        profile.update(
            driver="GTiff",
            width=window.width,
            height=window.height,
            transform=transform,
            compress="deflate",
            predictor=2,
            tiled=True,
            blockxsize=256,
            blockysize=256,
        )
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data, 1)
            dst.update_tags(**src.tags(), source=href)

    record["bytes"] = Path(out_path).stat().st_size
    return record


def fetch_mtl(href, out_path):
    url = urlparse(href)
    client = boto3.Session(region_name=S3_REGION).client("s3")
    response = client.get_object(Bucket=url.netloc, Key=url.path.lstrip("/"), RequestPayer="requester")
    Path(out_path).write_bytes(response["Body"].read())

In [ ]:
def strip_scheme(path):
    return path.removeprefix("s3://")


def is_done(previous, key, existing):
    """Should a band be skipped on this run? no_data results are final; ok results also need the file to still exist."""
    if previous is None:
        return False
    return previous["status"] == "no_data" or (previous["status"] == "ok" and strip_scheme(key) in existing)


def crop_scene(item, existing):
    """
    Crop all bands of a scene and upload them with its MTL.json, skipping finished work.

    A <scene_id>_record.json next to the rasters holds the result for each band and is what makes runs resumable.
    Returns (band records, number of network reads made).
    """
    scene_id = item.id
    prefix = f"{OUT_ROOT}/{scene_id}"
    record_key = f"{prefix}/{scene_id}_record.json"
    previous = {}
    if strip_scheme(record_key) in existing:
        previous = {row["band"]: row for row in json.loads(fs.cat(record_key))}

    rows = []
    n_reads = 0
    data_ok = False  # has any data band got valid pixels?
    data_error = False

    def attempt(band, key, work):
        """Run work() for one band, recording the time taken and any failure rather than stopping the scene."""
        nonlocal n_reads
        n_reads += 1
        start = time.perf_counter()
        try:
            record = work()
        except Exception as error:
            record = {"status": "error", "reason": f"{type(error).__name__}: {error}"[:500]}
        rows.append({"scene_id": scene_id, "band": band, "key": key, "seconds": time.perf_counter() - start, **record})
        return rows[-1]

    with tempfile.TemporaryDirectory() as tmp, rio_env():
        for band in bands_for(item):
            key = f"{prefix}/{scene_id}_{band}.tif"
            local = f"{tmp}/{scene_id}_{band}.tif"
            is_qa = band in QA_BANDS

            if is_done(previous.get(band), key, existing):
                row = previous[band]
                rows.append(row)
            elif is_qa and not data_ok:
                # QA bands are only worth keeping if the scene has data here. Don't record no_data as final while
                # a data-band failure could still turn out to be the reason.
                row = {
                    "scene_id": scene_id,
                    "band": band,
                    "key": key,
                    "seconds": 0.0,
                    "status": "error" if data_error else "no_data",
                    "reason": "data_band_failed" if data_error else "no_valid_data_bands",
                }
                rows.append(row)
            else:
                def work():
                    record = crop_band(s3_href(item.assets[band]), local, keep_empty=is_qa)
                    if record["status"] == "ok":
                        fs.put_file(local, key)
                    return record

                row = attempt(band, key, work)

            if not is_qa:
                data_ok = data_ok or row["status"] == "ok"
                data_error = data_error or row["status"] == "error"

        # MTL.json is kept for every scene, since Level 1 DNs can't be calibrated without it
        mtl_key = f"{prefix}/{scene_id}_MTL.json"
        if is_done(previous.get(MTL_ASSET), mtl_key, existing):
            rows.append(previous[MTL_ASSET])
        else:
            def work():
                local = f"{tmp}/{scene_id}_MTL.json"
                fetch_mtl(s3_href(item.assets[MTL_ASSET]), local)
                fs.put_file(local, mtl_key)
                return {"status": "ok", "bytes": Path(local).stat().st_size}

            attempt(MTL_ASSET, mtl_key, work)

    if n_reads or len(previous) != len(rows):
        with fs.open(record_key, "w") as f:
            json.dump(rows, f)
    return rows, n_reads


def safe_crop_scene(item, existing):
    """Keep one scene's unexpected failure from stopping the whole run."""
    try:
        return crop_scene(item, existing)
    except Exception as error:
        reason = f"{type(error).__name__}: {error}"[:500]
        return [{"scene_id": item.id, "band": "*", "status": "error", "reason": reason, "seconds": 0.0}], 1

In [ ]:
def scene_status(rows):
    if any(row["status"] == "error" for row in rows):
        return "error"
    data_rows = [row for row in rows if row["band"] not in QA_BANDS and row["band"] != MTL_ASSET]
    return "ok" if any(row["status"] == "ok" for row in data_rows) else "no_data"


def scenes_table(items, records):
    """One row per catalogue scene. scene_status is not_run for scenes this run didn't include."""
    table = pd.DataFrame(
        [
            {
                "scene_id": item.id,
                "datetime": pd.Timestamp(item.datetime),
                "prefix": item.id[:4],
                "platform": item.properties["platform"],
                "wrs_path": item.properties["landsat:wrs_path"],
                "wrs_row": item.properties["landsat:wrs_row"],
                "processing_level": item.id.split("_")[1],
                "collection_category": item.properties.get("landsat:collection_category"),
                "cloud_cover": item.properties.get("eo:cloud_cover"),
                "sun_elevation": item.properties.get("view:sun_elevation"),
                "sun_azimuth": item.properties.get("view:sun_azimuth"),
                "scene_status": scene_status(records[item.id]) if item.id in records else "not_run",
            }
            for item in items
        ]
    )
    return table


def write_tables(records):
    with fs.open(f"{OUT_ROOT}/scenes.parquet", "wb") as f:
        scenes_table(all_items, records).to_parquet(f)
    bands = pd.DataFrame([row for rows in records.values() for row in rows])
    with fs.open(f"{OUT_ROOT}/bands.parquet", "wb") as f:
        bands.to_parquet(f)
    return bands


def run_all(items, n_workers=N_WORKERS, max_passes=MAX_PASSES):
    """Process scenes in a thread pool. Returns (bands table, number of network reads across all passes)."""
    records = {}
    total_reads = 0
    todo = list(items)
    start = time.perf_counter()

    for attempt_number in range(max_passes):
        if attempt_number:
            print(f"Retrying {len(todo)} scenes with failures (pass {attempt_number + 1} of {max_passes})")
            time.sleep(30 * attempt_number)

        fs.invalidate_cache()
        existing = {strip_scheme(path) for path in fs.find(OUT_ROOT)}
        with ThreadPoolExecutor(max_workers=n_workers) as pool:
            futures = {pool.submit(safe_crop_scene, item, existing): item for item in todo}
            for count, future in enumerate(as_completed(futures), start=1):
                rows, n_reads = future.result()
                records[futures[future].id] = rows
                total_reads += n_reads
                if count % 25 == 0:
                    print(f"  {count}/{len(todo)} scenes, {(time.perf_counter() - start) / 60:.1f} min")
                    write_tables(records)  # keep the catalogue current in case the run stops

        todo = [item for item in todo if scene_status(records[item.id]) == "error"]
        if not todo:
            break

    bands = write_tables(records)
    statuses = Counter(scene_status(rows) for rows in records.values())
    print(f"{dict(statuses)} in {(time.perf_counter() - start) / 60:.1f} min, {total_reads} network reads")
    if todo:
        print("Scenes still failing (re-run to retry):", [item.id for item in todo])
    return bands, total_reads

## 4. Check access before running

Fails here if the hub can't read the requester-pays bucket. The region matters because reads are only free of
transfer charges from within `us-west-2`.

In [ ]:
print("boto3 region:", boto3.Session().region_name, "| AWS_REGION:", os.environ.get("AWS_REGION"))
try:
    print("Identity:", boto3.client("sts", region_name=S3_REGION).get_caller_identity()["Arn"])
except Exception as error:
    print("Could not read identity:", type(error).__name__)

check_item = run_items[-1]
start = time.perf_counter()
with rio_env(), rasterio.open(s3_href(check_item.assets["red"])) as src:
    check_window = aoi_window(src)
    check_data = src.read(1, window=check_window)
    print(f"Read {check_data.shape} from {check_item.id} red in {time.perf_counter() - start:.1f} s")
    print(f"CRS {src.crs}, resolution {src.res}, COG blocks {src.block_shapes[0]}, valid fraction {(check_data != 0).mean():.2f}")

## 5. Run

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    item_collection.save_object(f"{tmp}/items.json")
    fs.put_file(f"{tmp}/items.json", f"{OUT_ROOT}/items.json")

bands, n_reads = run_all(run_items)
bands.groupby(["status", "band"]).size().unstack("status", fill_value=0)

## 6. Checks (test run)

**Correctness.** Each output's CRS, transform and shape must match the source window. For one band per scene, the
pixels must also be identical to a direct read from the source.

In [ ]:
def open_output(key):
    return MemoryFile(fs.cat(key))


def check_outputs(bands):
    items_by_id = {item.id: item for item in run_items}
    problems = []
    n_checked = n_pixels_checked = 0
    tif_rows = bands[(bands.status == "ok") & (bands.band != MTL_ASSET)]

    with rio_env():
        for row in tif_rows.itertuples():
            with rasterio.open(s3_href(items_by_id[row.scene_id].assets[row.band])) as src, open_output(row.key) as mem, mem.open() as out:
                window = aoi_window(src)
                if out.crs != src.crs:
                    problems.append((row.key, "crs"))
                if out.transform != src.window_transform(window):
                    problems.append((row.key, "transform"))
                if (out.height, out.width) != (window.height, window.width):
                    problems.append((row.key, "shape"))
                if out.dtypes[0] != src.dtypes[0] or out.nodata != src.nodata:
                    problems.append((row.key, "dtype or nodata"))
                n_checked += 1
                if row.band == "red" and not np.array_equal(out.read(1), src.read(1, window=window)):
                    problems.append((row.key, "pixels"))
                elif row.band == "red":
                    n_pixels_checked += 1

    print(f"{n_checked} outputs checked for metadata, {n_pixels_checked} red bands checked pixel by pixel")
    return problems


problems = check_outputs(bands)
assert not problems, problems
print("No problems found")

**Pixel grid.** Do all the scenes share one pixel lattice at each resolution? A single row per resolution means a
later stack needs no resampling. Several rows mean scenes are offset by sub-pixel amounts and a stack would need
resampling to one grid.

In [ ]:
tifs = bands[(bands.status == "ok") & (bands.band != MTL_ASSET)]
lattice = (
    tifs.assign(x_phase=(tifs.x_origin % tifs.res).round(3), y_phase=(tifs.y_origin % tifs.res).round(3))
    .groupby(["res", "x_phase", "y_phase"])
    .agg(n_bands=("band", "size"), n_scenes=("scene_id", "nunique"))
)
lattice

**Coverage.** Every expected band of every test scene must have a row, and no band should have been left in an
error state.

In [ ]:
expected = {(item.id, band) for item in run_items for band in [*bands_for(item), MTL_ASSET]}
recorded = set(zip(bands.scene_id, bands.band))
assert expected == recorded, f"missing {expected - recorded}, unexpected {recorded - expected}"

scene_summary = scenes_table(run_items, {sid: rows.to_dict("records") for sid, rows in bands.groupby("scene_id")})
display_cols = ["scene_id", "datetime", "prefix", "processing_level", "cloud_cover", "sun_elevation", "scene_status"]
print(bands[bands.status != "ok"][["scene_id", "band", "status", "reason"]].to_string())
scene_summary[display_cols]

**Resume.** A second run must make no network reads. After deleting one output, a run must fetch only that file.

In [ ]:
assert TEST, "this cell deletes an output, so it only runs on the test prefix"

_, reads_second_run = run_all(run_items)
assert reads_second_run == 0, f"expected no reads, made {reads_second_run}"

victim = tifs.iloc[0].key
fs.rm(victim)
_, reads_after_delete = run_all(run_items)
assert reads_after_delete == 1, f"expected 1 read, made {reads_after_delete}"
fs.invalidate_cache()
assert fs.exists(victim)
print("Resume works: 0 reads on re-run, 1 read after deleting", victim)

**Quicklook.** One panel per test scene, using the finest band with data. Check that the glacier falls inside the
AOI (red outline), that SLC-off stripes show up on the later ETM+ scene, and how the night scene behaves.

In [ ]:
def quicklook(bands):
    summary = scene_summary.set_index("scene_id")
    fig, axes = plt.subplots(3, 5, figsize=(18, 11), constrained_layout=True)
    axes = axes.ravel()

    for ax, item in zip(axes, run_items):
        rows = bands[(bands.scene_id == item.id) & (bands.status == "ok")].set_index("band")
        info = summary.loc[item.id]
        title = f"{item.id[:4]} {item.datetime:%Y-%m-%d}\nsun {info.sun_elevation:.0f}°, {info.scene_status}"
        band = next((b for b in ["pan", "red", "lwir11", "lwir"] if b in rows.index), None)
        if band is None:
            ax.text(0.5, 0.5, "no raster", ha="center", transform=ax.transAxes)
        else:
            with open_output(rows.loc[band, "key"]) as mem, mem.open() as src:
                data = src.read(1, masked=True).astype(float).filled(np.nan)
                extent = plotting_extent(src)
            if np.isfinite(data).any():
                low, high = np.nanpercentile(data, [2, 98])
                ax.imshow(data, extent=extent, cmap="gray", vmin=low, vmax=high)
            title += f"\n{band} ({rows.loc[band, 'res']:.0f} m, {rows.loc[band, 'valid_frac']:.0%} valid)"
        ax.plot(AOI_X, AOI_Y, color="red", linewidth=1)
        ax.set_title(title, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect("equal")

    for ax in axes[len(run_items):]:
        ax.axis("off")
    return fig


quicklook(bands);

## 7. Projection for the full run

Time and size per scene are averaged by ID prefix, then scaled by the number of scenes of each prefix in the full
catalogue. The request count is an upper bound: two header reads per band plus one per COG block touched. It assumes
S3 GET requests at $0.0004 per 1,000. Check `nasa-cryo-scratch` retention before relying on it for the full output.

**Rule:** stay with the thread pool unless the projected time exceeds about 4 hours. Above that, map `crop_scene`
over Dask workers.

In [ ]:
def project_full_run(bands):
    tif_rows = bands[(bands.status == "ok") & (bands.band != MTL_ASSET)]
    per_scene = (
        bands.groupby("scene_id")  # every band counts for time, since an empty crop still takes a read
        .agg(seconds=("seconds", "sum"), bytes=("bytes", "sum"))
        .join(tif_rows.groupby("scene_id").agg(blocks=("blocks", "sum"), n_bands=("band", "size")))
    )
    per_scene["prefix"] = per_scene.index.str[:4]
    mean_by_prefix = per_scene.groupby("prefix")[["seconds", "bytes", "blocks", "n_bands"]].mean()

    counts = pd.Series(Counter(item.id[:4] for item in all_items), name="n_scenes")
    table = mean_by_prefix.join(counts, how="outer")
    if table.isna().any().any():
        print("WARNING: some prefixes were not measured, so these totals are too low:", table[table.isna().any(axis=1)].index.tolist())

    total_seconds = (table.seconds * table.n_scenes).sum()
    total_gb = (table.bytes * table.n_scenes).sum() / 1e9
    requests = ((table.blocks + 2 * table.n_bands) * table.n_scenes).sum()
    hours = total_seconds / N_WORKERS / 3600

    print(f"Scenes:             {table.n_scenes.sum():.0f}")
    print(f"Output size:        {total_gb:.1f} GB (compressed)")
    print(f"Time:               {hours:.1f} h on {N_WORKERS} threads, if throughput scales from this test")
    print(f"S3 GET requests:    at most {requests:,.0f}, about ${requests / 1000 * 0.0004:.2f}")
    print("Decision:           " + ("consider Dask workers" if hours > 4 else "thread pool on the hub is enough"))
    return table


project_full_run(bands)

## Next steps

Once these results look right: set `TEST = False`, restart the kernel, and run sections 1-5. Then copy the output
to `remote sensing/landsat/flask_region/` before scratch retention removes it.

Every scene is Tier 2, and L1GS scenes aren't terrain-corrected, so scenes will be misaligned to some degree.
Co-register them before any change or velocity analysis.

# Part 2: COGs, browse images and viewer

**Start here if the download is already on scratch.** Run the two code cells in section 1, with `TEST` set the same
as it was for the download, then skip to the next cell. Sections 2-7 don't need re-running.

- Nothing is read from USGS, so there are no requester-pays requests.
- The original GeoTIFFs aren't changed. Everything new goes in a separate `<OUT_ROOT>_cog` folder.
- Every conversion is compared with its source before upload, and any re-run skips finished files.
- `flask_landsat_viewer.html` must be next to this notebook.

## 8. Check what's on scratch

In [ ]:
import io

from PIL import Image, features
from rasterio.io import MemoryFile
from rasterio.shutil import copy as rio_copy

assert features.check("webp"), "this Pillow build can't write WebP"

COG_ROOT = f"{OUT_ROOT}_cog"
VIEWER_HTML = "flask_landsat_viewer.html"
assert Path(VIEWER_HTML).exists(), f"{VIEWER_HTML} needs to be next to this notebook"


def bare(path):
    return path.removeprefix("s3://")


def read_table(name):
    with fs.open(f"{OUT_ROOT}/{name}", "rb") as f:
        return pd.read_parquet(f)


fs.invalidate_cache()
scenes = read_table("scenes.parquet")
bands = read_table("bands.parquet")
processed = scenes[scenes.scene_status != "not_run"]
is_raster = (bands.status == "ok") & (bands.band != MTL_ASSET)

print(f"Reading from: {OUT_ROOT}")
print(f"Writing to:   {COG_ROOT}")
print(f"{len(processed)} of {len(scenes)} catalogue scenes processed: {processed.scene_status.value_counts().to_dict()}")
print(f"{is_raster.sum()} rasters, {(bands.band == MTL_ASSET).sum()} MTL files")
if len(processed) < len(scenes):
    print("NOTE: not every scene has been downloaded here. If you ran the full download, check that TEST = False.")
if (bands.status == "error").any():
    print("WARNING: some bands failed during the download and won't be included:")
    print(bands[bands.status == "error"][["scene_id", "band", "reason"]].to_string())

## 9. Convert the GeoTIFFs to COGs

Deflate-compressed with 256 px tiles and overviews. QA bands use nearest-neighbour overviews, because averaging
would corrupt their bit flags. Full-resolution pixels are unchanged, and a file is only uploaded after that's been
checked.

In [ ]:
def cog_problems(plain_path, cog_path):
    """Differences between a source GeoTIFF and its COG, plus anything that makes it not a valid COG."""
    problems = []
    with rasterio.open(plain_path) as plain, rasterio.open(cog_path) as cog:
        if cog.tags(ns="IMAGE_STRUCTURE").get("LAYOUT") != "COG":
            problems.append("not a COG")
        if min(cog.width, cog.height) > 512 and not cog.overviews(1):
            problems.append("no overviews")
        for name in ("crs", "transform", "width", "height", "dtypes", "nodata"):
            if getattr(plain, name) != getattr(cog, name):
                problems.append(name)
        if not np.array_equal(plain.read(1), cog.read(1)):
            problems.append("pixels")
    return problems


def convert_band(row, existing):
    """Convert one raster unless it's already done. Returns (dest key, size in bytes, error message or None)."""
    dest = row.key.replace(OUT_ROOT, COG_ROOT, 1)
    if bare(dest) in existing:
        return dest, existing[bare(dest)], None
    try:
        with tempfile.TemporaryDirectory() as tmp:
            plain_path, cog_path = f"{tmp}/plain.tif", f"{tmp}/cog.tif"
            Path(plain_path).write_bytes(fs.cat(row.key))
            rio_copy(
                plain_path,
                cog_path,
                driver="COG",
                compress="DEFLATE",
                predictor="YES",
                blocksize=256,
                overview_resampling="nearest" if row.band in QA_BANDS else "average",
            )
            problems = cog_problems(plain_path, cog_path)
            if problems:
                raise ValueError(f"COG differs from source: {problems}")
            size = Path(cog_path).stat().st_size
            fs.put_file(cog_path, dest)
        return dest, size, None
    except Exception as error:
        return dest, None, f"{type(error).__name__}: {error}"[:500]


def existing_sizes(root):
    fs.invalidate_cache()
    return {path: info["size"] for path, info in fs.find(root, detail=True).items()}


def convert_all(bands, n_workers=N_WORKERS):
    """Convert every raster and copy the MTL files. Returns the bands table pointing at the new files, and the failures."""
    existing = existing_sizes(COG_ROOT)
    rasters = bands[(bands.status == "ok") & (bands.band != MTL_ASSET)]
    start = time.perf_counter()

    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        futures = {pool.submit(convert_band, row, existing): row.Index for row in rasters.itertuples()}
        results = {}
        for count, future in enumerate(as_completed(futures), start=1):
            results[futures[future]] = future.result()
            if count % 500 == 0:
                print(f"  {count}/{len(rasters)} rasters, {(time.perf_counter() - start) / 60:.1f} min")

    # Copy metadata as is
    mtl_rows = bands[(bands.status == "ok") & (bands.band == MTL_ASSET)]

    def copy_mtl(key):
        dest = key.replace(OUT_ROOT, COG_ROOT, 1)
        if bare(dest) not in existing:
            fs.cp_file(key, dest)
        return dest

    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        mtl_dests = dict(zip(mtl_rows.index, pool.map(copy_mtl, mtl_rows.key)))

    updated = bands.copy()
    for index, (dest, size, error) in results.items():
        updated.loc[index, "key"] = dest
        updated.loc[index, "bytes"] = size
        if error:
            updated.loc[index, "status"] = "error"
            updated.loc[index, "reason"] = error
    for index, dest in mtl_dests.items():
        updated.loc[index, "key"] = dest

    with fs.open(f"{COG_ROOT}/bands.parquet", "wb") as f:
        updated.to_parquet(f)
    for name in ("scenes.parquet", "items.json"):
        if fs.exists(f"{OUT_ROOT}/{name}"):
            fs.cp_file(f"{OUT_ROOT}/{name}", f"{COG_ROOT}/{name}")

    failures = {bands.loc[i, "key"]: error for i, (_, _, error) in results.items() if error}
    print(f"{len(results) - len(failures)} of {len(results)} rasters are COGs, in {(time.perf_counter() - start) / 60:.1f} min")
    return updated, failures


cog_bands, failures = convert_all(bands)
was_raster = (bands.status == "ok") & (bands.band != MTL_ASSET)
print(f"GeoTIFFs {bands[was_raster].bytes.sum() / 1e9:.2f} GB -> COGs {cog_bands[was_raster].bytes.sum() / 1e9:.2f} GB")
if failures:
    print("FAILED (re-run this cell to retry):")
    for key, error in failures.items():
        print(" ", key, error)

## 10. Browse images and viewer

For cycling through about 1,200 scenes, each one gets an 8-bit WebP of every data band at its native resolution, plus
an RGB composite that the viewer opens on. MSS has no blue band, so its composite is NIR/red/green, which the viewer
labels as false colour. Everything is stretched per scene, so brightness and colour can't be compared between scenes.
Composites are for looking at, not measuring: bands often saturate over snow, and clipped channels are filled in from
the others (see `render_composite`). Choose a single band in the viewer to see the band as it is, and use the COGs for
anything quantitative. `index.json` lists every scene, including empty ones, for the viewer page.

In [ ]:
BROWSE_BANDS = ["pan", "red", "lwir11", "lwir"]
COMPOSITES = [("red", "green", "blue"), ("nir08", "red", "green")]
GRID_COLUMNS = ["res", "width", "height", "x_origin", "y_origin"]
BROWSE_QUALITY = 80


def read_band(cog_bytes):
    """Read a band from COG bytes. Returns it with a mask of its valid pixels."""
    with MemoryFile(cog_bytes) as mem, mem.open() as src:
        data = src.read(1)
        nodata = src.nodata if src.nodata is not None else 0
    return data, data != nodata


def encode_webp(red, green, blue, valid):
    buffer = io.BytesIO()
    rgba = np.dstack([red, green, blue, valid.astype("uint8") * 255])
    Image.fromarray(rgba, "RGBA").save(buffer, "WEBP", quality=BROWSE_QUALITY, method=4)
    return buffer.getvalue()


def render_browse(cog_bytes):
    """Turn COG bytes into greyscale WebP bytes, with nodata pixels transparent. A fully saturated band comes out white."""
    data, valid = read_band(cog_bytes)
    low, high = np.percentile(data[valid], [2, 98])
    gray = np.where(data >= high, 255, np.clip((data.astype("float32") - low) / max(high - low, 1) * 255, 0, 255)).astype("uint8")
    return encode_webp(gray, gray, gray, valid)


def render_composite(red_bytes, green_bytes, blue_bytes):
    """Turn three COGs into colour WebP bytes, with pixels transparent unless all three are valid.

    A separate stretch per channel tints the ice, because bands saturate over snow at different levels: in real scenes
    up to 83% of a channel sits at its maximum. So channels are balanced on the pixels where none is clipped, and a
    clipped value is replaced by the mean of the channels that aren't, or white if all three are clipped.
    """
    bands, masks = zip(*map(read_band, (red_bytes, green_bytes, blue_bytes)))
    valid = np.logical_and.reduce(masks)
    clipped = [band == band[valid].max() for band in bands]
    usable = valid & ~np.logical_or.reduce(clipped)
    if usable.sum() < 0.01 * valid.sum():
        usable = valid

    scaled = []
    for band in bands:
        low, middle = np.percentile(band[usable], [2, 50])
        scaled.append((band.astype("float32") - low) / max(middle - low, 1))

    n_unclipped = np.sum([~c for c in clipped], axis=0)
    total = np.sum([np.where(c, 0, s) for s, c in zip(scaled, clipped)], axis=0)
    fill = np.full(valid.shape, np.inf, dtype="float32")
    np.divide(total, n_unclipped, out=fill, where=n_unclipped > 0)
    filled = [np.where(c, fill, s) for s, c in zip(scaled, clipped)]

    brightness = np.mean(filled, axis=0)[valid]
    finite = brightness[np.isfinite(brightness)]
    high = max(np.percentile(finite, 98), 1) if finite.size else 1
    return encode_webp(*[(np.clip(f / high, 0, 1) * 255).astype("uint8") for f in filled], valid)


def data_bands(scene_rows):
    """A scene's bands with data, indexed by band name."""
    ok = scene_rows[(scene_rows.status == "ok") & ~scene_rows.band.isin(QA_BANDS | {MTL_ASSET})]
    return ok.set_index("band", drop=False)


def browse_band(ok):
    """The row of the single band to show when there's no composite: the first of BROWSE_BANDS, then any, or None."""
    preferred = [band for band in BROWSE_BANDS if band in ok.index]
    if preferred:
        return ok.loc[preferred[0]]
    return ok.iloc[0] if len(ok) else None


def composite_bands(ok):
    """The first of COMPOSITES whose three bands all have data on one pixel grid, or None."""
    for names in COMPOSITES:
        if set(names) <= set(ok.index) and len(ok.loc[list(names), GRID_COLUMNS].drop_duplicates()) == 1:
            return list(names)
    return None


def number_or_none(value):
    return None if pd.isna(value) else float(value)  # NaN isn't valid JSON, and would stop the viewer loading


def image_info(rows):
    """The index.json record of an image made from these rows, which share a pixel grid."""
    row = rows[0]
    res = float(row.res)
    return {
        "res": res,
        "valid": round(min(float(r.valid_frac) for r in rows), 3),
        "bounds": [
            float(row.x_origin),
            float(row.y_origin) - float(row.height) * res,
            float(row.x_origin) + float(row.width) * res,
            float(row.y_origin),
        ],
    }


def browse_entry(scene, scene_rows, existing):
    """Write one scene's browse images if needed. Returns its index.json entry and a list of errors."""
    entry = {
        "id": scene.scene_id,
        "datetime": scene.datetime.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "platform": scene.platform,
        "path": scene.wrs_path,
        "row": scene.wrs_row,
        "level": scene.processing_level,
        "cloud": number_or_none(scene.cloud_cover),
        "sun": number_or_none(scene.sun_elevation),
        "images": {},
        "composite": None,
        "default": None,
    }
    if scene_rows is None:
        return entry, []
    ok = data_bands(scene_rows)
    errors = []

    def add_image(name, rows, render):
        key = f"{COG_ROOT}/browse/{name}/{scene.scene_id}.webp"
        try:
            if bare(key) not in existing:
                fs.pipe_file(key, render(*[fs.cat(row.key) for row in rows]))
            entry["images"][name] = image_info(rows)
        except Exception as error:
            errors.append(f"{scene.scene_id} {name}: {type(error).__name__}: {error}"[:500])

    names = composite_bands(ok)
    if names:
        add_image("rgb", [ok.loc[name] for name in names], render_composite)
        entry["composite"] = names if "rgb" in entry["images"] else None
    for row in ok.itertuples():
        add_image(row.band, [row], render_browse)

    fallback = browse_band(ok[ok.band.isin(list(entry["images"]))])
    entry["default"] = None if fallback is None else fallback.band
    return entry, errors


def build_viewer(processed, cog_bands, n_workers=N_WORKERS):
    """Write the browse images, index.json and the viewer page. Returns the index."""
    existing = existing_sizes(COG_ROOT)

    # An earlier version of this section wrote one image per scene directly in browse/
    browse_prefix = bare(f"{COG_ROOT}/browse/")
    old_layout = [path for path in existing if path.startswith(browse_prefix) and "/" not in path[len(browse_prefix):]]
    if old_layout:
        fs.rm(old_layout)
        print(f"Removed {len(old_layout)} browse images in the old layout")

    rows_by_scene = dict(tuple(cog_bands.groupby("scene_id")))
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        results = list(pool.map(lambda scene: browse_entry(scene, rows_by_scene.get(scene.scene_id), existing), processed.itertuples()))

    index = {
        "title": "Flask Glacier Landsat record",
        "product": "Landsat Collection 2 Level 1 (DN, stretched per scene)",
        "crs": "EPSG:3031",
        "aoi_bounds": list(AOI_BOUNDS),
        "aoi_vertices": [list(vertex) for vertex in AOI_VERTICES_3031],
        "cogs_available": True,
        "scenes": [entry for entry, _ in results],
    }
    with fs.open(f"{COG_ROOT}/index.json", "w") as f:
        json.dump(index, f, separators=(",", ":"), allow_nan=False)
    fs.put_file(VIEWER_HTML, f"{COG_ROOT}/index.html")

    errors = [error for _, scene_errors in results for error in scene_errors]
    scenes = index["scenes"]
    print(f"{sum(bool(e['images']) for e in scenes)} of {len(scenes)} scenes have images, {sum('rgb' in e['images'] for e in scenes)} with a composite")
    print(f"{sum(len(e['images']) for e in scenes)} browse images in all")
    if errors:
        print("FAILED (re-run this cell to retry):", *errors, sep="\n  ")
    return index


index = build_viewer(processed, cog_bands)
sizes = existing_sizes(COG_ROOT)
browse_mb = sum(size for path, size in sizes.items() if path.endswith(".webp")) / 1e6
print(f"Browse images {browse_mb:.0f} MB. Everything in {COG_ROOT}: {sum(sizes.values()) / 1e9:.2f} GB")

## 11. Share (optional)

`nasa-cryo-scratch` isn't public, so colleagues can't reach anything there. This copies the whole `_cog` folder to a
bucket they can read, and the viewer link needs no login.

**Check before setting `PUBLISH = True`:**
- `ldeo-glaciology` is publicly listable, so everything copied there is readable by anyone. Landsat data is public
  domain, so that's fine for these files.
- Public downloads are billed to the bucket owner.
- It needs the bucket's service-account key on the hub. Upload it to your home folder, outside this repo, so it
  can't be committed.

In [ ]:
PUBLISH = False
PUBLISH_ROOT = "gs://ldeo-glaciology/landsat/flask_region"
GCS_KEY_FILE = "~/ldeo-glaciology-key.json"

# Without these the viewer page would download rather than open
CONTENT_TYPES = {
    ".html": "text/html; charset=utf-8",
    ".json": "application/json",
    ".webp": "image/webp",
    ".tif": "image/tiff; application=geotiff; profile=cloud-optimized",
}
ALWAYS_COPY = {"index.html", "index.json", "bands.parquet", "scenes.parquet"}


def publish(dest_fs, dest_root, n_workers=N_WORKERS):
    """Copy everything under COG_ROOT to dest_root, skipping files already there. Returns the number copied."""
    fs.invalidate_cache()
    prefix = bare(COG_ROOT) + "/"
    relative = [path[len(prefix):] for path in fs.find(COG_ROOT)]
    already = set(dest_fs.find(dest_root))
    bare_dest = dest_root.removeprefix("gs://")
    todo = [rel for rel in relative if rel in ALWAYS_COPY or f"{bare_dest}/{rel}" not in already]

    def copy_one(rel):
        content_type = CONTENT_TYPES.get(Path(rel).suffix, "application/octet-stream")
        dest_fs.pipe_file(f"{dest_root}/{rel}", fs.cat(f"{COG_ROOT}/{rel}"), content_type=content_type)

    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        list(pool.map(copy_one, todo))
    return len(todo)


if PUBLISH:
    import gcsfs

    with open(os.path.expanduser(GCS_KEY_FILE)) as key_file:
        gcs = gcsfs.GCSFileSystem(token=json.load(key_file))
    copied = publish(gcs, PUBLISH_ROOT)
    print(f"Copied {copied} files.\nViewer: https://storage.googleapis.com/{PUBLISH_ROOT.removeprefix('gs://')}/index.html")
else:
    print("Not published. Set PUBLISH = True to copy to", PUBLISH_ROOT)